In [4]:
!pip install osmnx
!pip install networkx
!pip install folium
import osmnx as ox
import networkx as nx
import folium

place_name = "Mumbai, India"


# Define Two Coordinates from Gateway of India to Siddhivinayak Temple
origin_point = (18.92198, 72.83465)
destination_point = (19.01699, 72.83040)

G = ox.graph_from_point(origin_point, dist=15000, network_type="drive")
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

# Find the nearest nodes
orig_node = ox.distance.nearest_nodes(G, X=origin_point[1], Y=origin_point[0])
dest_node = ox.distance.nearest_nodes(G, X=destination_point[1], Y=destination_point[0])

# Calculate Fastest Path
route_time = nx.shortest_path(G, orig_node, dest_node, weight="travel_time")

# Extract Metrics
dist_m = nx.shortest_path_length(G, orig_node, dest_node, weight="length")
time_sec = nx.shortest_path_length(G, orig_node, dest_node, weight="travel_time")

print(f"Shortest Path Distance: {dist_m / 1000:.2f} km")
print(f"Estimated Travel Time: {time_sec / 60:.2f} minutes")

# Convert the route nodes to a GeoDataFrame of the path's edges and creating basemap
route_edges = ox.routing.route_to_gdf(G, route_time)
m = route_edges.explore(
    color="blue",
    style_kwds={'weight': 5, 'opacity': 0.8},
    name="Fastest Route"
)

# Add markers for Start and End
folium.Marker(location=origin_point, popup="Gateway of India", icon=folium.Icon(color='green')).add_to(m)
folium.Marker(location=destination_point, popup="Siddhivinayak Temple", icon=folium.Icon(color='red')).add_to(m)

m

Shortest Path Distance: 11.40 km
Estimated Travel Time: 14.28 minutes


In [5]:
import osmnx as ox
import folium

place_name = "Mumbai City, Maharashtra, India"

# Download Networks
print(f"Fetching networks for {place_name}...")
G_drive = ox.graph_from_place(place_name, network_type='drive')
G_walk = ox.graph_from_place(place_name, network_type='walk')

# Download Buildings
print("Fetching building footprints...")
buildings = ox.features_from_place(place_name, tags={'building': True})

# Project to EPSG:7781 which is specific for Maharashtra/Mumbai
buildings_proj = ox.projection.project_gdf(buildings, to_crs="EPSG:7781")

# Calculate Building Area
total_building_area = buildings_proj.area.sum()

# Get the total land area using the convex hull of the buildings
total_land_area = buildings_proj.unary_union.convex_hull.area

# Print Statistics
print(f"\n--- Statistics for {place_name} ---")
print(f"Total Building Area: {total_building_area:,.2f} m²")
print(f"Total Study Area: {total_land_area:,.2f} m²")
print(f"Building Coverage Ratio (BCR): {(total_building_area / total_land_area) * 100:.2f}%")

m = buildings.explore(
    column="building",  # Color by building type if available
    cmap="viridis",
    name="Buildings",
    style_kwds={'fillOpacity': 0.7}
)

folium.LayerControl().add_to(m)
m

Output hidden; open in https://colab.research.google.com to view.